# Week 0 — Programming *with* an LLM (not chatting with one)

**Self-paced.** Budget 90–120 minutes, plus download time for the model.

**Instructions and what to hand in:** [`START-HERE.md`](START-HERE.md)

---

## Why this exists

In this semester, you will build classical AI algorithms and then **measure them
against an LLM doing the same task**. That comparison is only worth anything if the
LLM side is done properly.

Chatting with a model and *programming* with one are different skills. Chatting is
interactive, forgiving, and unreproducible. Programming is batched, cached, parsed,
and verified. You are going to run 120 instances and put the numbers in a report 
someone will challenge.

By the end of this notebook you will have:

1. A working LLM backend (local, manual, or your own key): **whichever you can get**
2. A cache, so you never pay for the same call twice
3. A **parser** that turns prose into data, and a **verifier** that checks it
4. Your first measured comparison, with a scaling curve
5. One documented failure, posted to the Failure Atlas

> ### The one idea to carry out of Week 0
>
> **An unreliable generator plus a sound verifier can be a reliable system.**
>
> You will meet this in weeks 4, 6, 7, 8, 9, and 11. Today you build the verifier
> half, for your own experiments.

## 0 · Check your environment

If this cell fails, stop and work through
[`START-HERE.md`](START-HERE.md) first. The `!` prefix runs a shell
command from inside Jupyter.

In [1]:
import sys, subprocess
r = subprocess.run([sys.executable, "-m", "aicourse.doctor"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
print("exit code:", r.returncode, "(0 = all required checks passed)")

CMP-4004 environment check
----------------------------------------------------------
✔ Python 3.12.0
✗ matplotlib is missing — pip install -r requirements.txt
✔ numpy 2.4.2, pytest 8.2.0
✔ optional: pandas 2.2.3, sklearn MISSING, networkx MISSING, pyperplan MISSING
! SWI-Prolog not found — needed for weeks 8-9 only
    brew install swi-prolog   /   sudo apt install swi-prolog
✔ backend ollama  serving (1 model(s))
✔ backend manual  always available
✔ backend echo    always available (testing only — not a model)
  probing ollama (qwen2.5:3b) ...
✔ LLM backend: ollama (qwen2.5:3b) — responded in 13.1 s: 'ready'
✔ .llm_cache/ writable (24 entries)
----------------------------------------------------------
→ Some REQUIRED checks failed (marked ✗). Fix those first.
  Lines marked ! are warnings and will not block you this week.

exit code: 1 (0 = all required checks passed)


### What "Ready" does and does not mean

- `✗` marks a **required** failure. Fix those.
- `!` marks a warning. Missing SWI-Prolog is fine until week 8; missing Ollama is
  fine forever if you use the manual backend.
- **`LLM backend: manual` is a PASS.** Every lab in this course is completable by
  pasting prompts into any chat interface. You report a smaller *n* and say so.
  You are not penalized.

Paste this output into your Week 0 submission.

## 1 · Your first programmatic call

Three lines. Note what is *not* here: no chat window, no scrolling, no copy-paste.

In [2]:
from aicourse import LLM

llm = LLM(backend="auto")          # picks ollama > api > manual
AUTOMATED_EARLY = llm.backend in ("ollama", "api")
print("backend:", llm.backend, "| model:", llm.model)

r = llm.complete("In one sentence: what is a heuristic in search?")
if r.error:
    print(f"\n[no response: {r.error}]")
    print("\nIf that mentions 'not a TTY', you ran this with Run All on the")
    print("manual backend. Run this cell on its own so you can paste an answer,")
    print("or set up Ollama. Either is fine.")
else:
    print("\n" + r.text.strip()[:400])
    print(f"\n[{r.elapsed:.1f}s, cached={r.cached}]")

backend: ollama | model: qwen2.5:3b

A heuristic in search is a strategy used to guide the search process towards a solution more efficiently, often by making informed, albeit approximate, decisions.

[0.0s, cached=True]


### If you are on the manual backend

The cell above printed a prompt and waited for you to paste a response ending with
a line containing only `END`. That is the workflow all semester. It is slower, and
it works.

To try the mechanics without a model at all, use the `echo` backend. A fake that
returns a fixed string. **Never report results from `echo`.**

In [3]:
fake = LLM(backend="echo")
print(fake.complete("anything").text)

[echo backend] 8 chars received. This is NOT a model response.


## 2 · The cache is not an optimization. It is your evidence.

Run the same prompt twice and watch the second call cost nothing.

In [4]:
import time

probe = "Name the author of the 1950 paper 'Computing Machinery and Intelligence'."

# Use whichever backend is available -- the caching behaviour is identical.
demo = llm if AUTOMATED_EARLY else LLM(backend="echo")
t0 = time.perf_counter(); a = demo.complete(probe); t1 = time.perf_counter()
b = demo.complete(probe); t2 = time.perf_counter()

print(f"first  call: {t1-t0:7.3f}s   cached={a.cached}")
print(f"second call: {t2-t1:7.3f}s   cached={b.cached}")
print(f"same text  : {a.text == b.text}")
print(f"\ncache: {demo.cache.stats()}")
if not AUTOMATED_EARLY:
    print("(shown with the echo backend; behaviour is identical for a real model)")

first  call:   0.001s   cached=True
second call:   0.000s   cached=True
same text  : True

cache: {'entries': 24, 'hits': 3, 'misses': 0, 'hit_rate': 1.0}


### Why this matters three times over

1. **Reproducibility**: a graded claim must be re-derivable. The cache *is* the
   proof you ran what you said you ran.
2. **Cost**: you will rewrite your analysis code ten times. Re-running 100 CPU
   inferences each time is hours you do not have.
3. **Honesty**: the cache is an audit trail. It makes *"we ran 30 instances"*
   checkable by a hostile reader.

**You commit `.llm_cache/` to your repo.** It is raw data, not build output.

### ⚠️ What the cache key includes

`SHA-256(backend, model, prompt, temperature, seed)`.

Change *any* of those and you get a real call. That is deliberate: a cache that
ignored temperature would silently serve you a greedy answer when you asked for a
sampled one, and you would report it as a sampled result.

In [5]:
from aicourse.cache import cache_key

base = cache_key("ollama", "qwen2.5:3b", "hello", 0.0, 0)
print("baseline           ", base[:16])
for label, args in [
    ("different prompt   ", ("ollama", "qwen2.5:3b", "hello!", 0.0, 0)),
    ("different temp     ", ("ollama", "qwen2.5:3b", "hello", 0.7, 0)),
    ("different seed     ", ("ollama", "qwen2.5:3b", "hello", 0.0, 1)),
    ("different model    ", ("ollama", "qwen2.5:1.5b", "hello", 0.0, 0)),
    ("identical          ", ("ollama", "qwen2.5:3b", "hello", 0.0, 0)),
]:
    k = cache_key(*args)
    print(f"{label} {k[:16]}   {'SAME' if k == base else 'different'}")

baseline            c7368d2235efb78f
different prompt    8bfaf3982b89be45   different
different temp      7fca231ece432e4a   different
different seed      910392812b058bf4   different
different model     e67a9a50fbe739a7   different
identical           c7368d2235efb78f   SAME


## 3 · ⚠️ The hard part: prose is not data

This is where most students lose a weekend, so we do it now.

An LLM returns **text**. Your experiment needs a **number**, a **list**, or a
**label**. The gap between those is where silent measurement errors live.

Watch a naive parser fail on outputs that all look fine to a human.

In [6]:
# Realistic variations on "answer with a number". None are unreasonable;
# all of them break a naive parser.
samples = [
    "4",
    "The answer is 4.",
    "4.0",
    "**4**",
    "Four.",
    "I think it's 4, but it depends on the encoding.",
    "```\n4\n```",
    "The answer is 4. Let me explain why: 2+2 means...",
    "Sure! Here you go: 42 is the answer to 2+2? No — 4.",
    "",
]

def parse_naive(text):
    """The parser everybody writes first."""
    return int(text.strip())

print(f"  {'output':<52}{'result'}")
print("  " + "-" * 68)
for s in samples:
    try:
        got = parse_naive(s)
    except Exception as exc:
        got = f"{type(exc).__name__}"
    print(f"  {s[:50]!r:<52}{got}")

  output                                              result
  --------------------------------------------------------------------
  '4'                                                 4
  'The answer is 4.'                                  ValueError
  '4.0'                                               ValueError
  '**4**'                                             ValueError
  'Four.'                                             ValueError
  "I think it's 4, but it depends on the encoding."   ValueError
  '```\n4\n```'                                       ValueError
  'The answer is 4. Let me explain why: 2+2 means...' ValueError
  'Sure! Here you go: 42 is the answer to 2+2? No — 4'ValueError
  ''                                                  ValueError


### The fix has three parts, in order of importance

1. **Constrain the output format in the prompt.** Cheapest and most effective.
2. **Parse defensively**, with an explicit "could not parse" outcome.
3. **Never silently coerce.** A `None` you counted is a data point. A `0` you
   invented is a lie in your results table.

In [7]:
import re

def parse_int(text):
    """Return (value, failure_mode). NEVER guesses."""
    if text is None or not text.strip():
        return None, "empty"
    # Prefer a fenced or final-line answer, which is what we asked for.
    fence = re.search(r"```(?:\w+)?\s*(-?\d+)\s*```", text)
    if fence:
        return int(fence.group(1)), None
    tagged = re.search(r"ANSWER\s*[:=]\s*(-?\d+)", text, re.IGNORECASE)
    if tagged:
        return int(tagged.group(1)), None
    # Match integers AND decimals, so "4.0" is one number rather than two.
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not nums:
        return None, "no-number-found"
    vals = []
    for tok in nums:
        f = float(tok)
        if f != int(f):
            return None, "not-an-integer"      # 4.5 is not an int answer
        vals.append(int(f))
    if len(set(vals)) > 1:
        # Two DIFFERENT numbers and no marker: we genuinely do not know which
        # one is the answer. Refusing to guess is the correct behaviour.
        return None, "ambiguous"
    return vals[0], None


print(f"  {'output':<52}{'value':>8}  failure")
print("  " + "-" * 76)
for s in samples:
    v, mode = parse_int(s)
    print(f"  {s[:50]!r:<52}{str(v):>8}  {mode or ''}")

  output                                                 value  failure
  ----------------------------------------------------------------------------
  '4'                                                        4  
  'The answer is 4.'                                         4  
  '4.0'                                                      4  
  '**4**'                                                    4  
  'Four.'                                                 None  no-number-found
  "I think it's 4, but it depends on the encoding."          4  
  '```\n4\n```'                                              4  
  'The answer is 4. Let me explain why: 2+2 means...'     None  ambiguous
  'Sure! Here you go: 42 is the answer to 2+2? No — 4'    None  ambiguous
  ''                                                      None  empty


### Read the `ambiguous` row carefully

`"Sure! Here you go: 42 is the answer to 2+2? No — 4."` contains both `42` and `4`.
A parser that grabbed the first number would score this **wrong**; one that grabbed
the last would score it **right**. Both are guesses dressed as measurements.

Returning `None, "ambiguous"` is the honest option: **it goes in your failure-mode
column, not your accuracy column.**

> **Your parser is part of your experiment.** If you tune it until the LLM looks
> good, you have measured your parser. Write it before you see the results, and say
> in your report how many outputs it failed to parse.

## 4 · Prompting as an engineering variable

Prompt wording changes results. That makes it a **variable you must control and
report**, exactly like a learning rate.

Four prompts for one task. Notice that the difference between them is entirely
about *making the output machine-readable*, not about politeness or persona.

In [8]:
TASK = "What is 17 * 23?"

PROMPTS = {
    "bare":        TASK,
    "format":      TASK + "\nReply with only the number.",
    "tagged":      TASK + "\nReply with exactly: ANSWER: <number>",
    "fenced+cot":  (TASK + "\nThink step by step, then give the final answer "
                    "in a fenced code block containing only the number."),
}

AUTOMATED = llm.backend in ("ollama", "api")

if AUTOMATED:
    print(f"  {'prompt style':<14}{'parsed':>9}{'failure':>16}   raw (truncated)")
    print("  " + "-" * 78)
    for name, p in PROMPTS.items():
        resp = llm.complete(p)
        val, mode = parse_int(resp.text)
        raw = resp.text.strip().replace("\n", " ")[:30]
        print(f"  {name:<14}{str(val):>9}{str(mode or ''):>16}   {raw!r}")
    print(f"\n  ground truth: {17*23}")
else:
    print("Skipped: needs an automated backend (ollama or api).\n")
    print("MANUAL BACKEND: do this by hand. Send all four prompts below to any")
    print("chat interface, paste each answer through parse_int(), and record the")
    print("table yourself. It is four prompts -- worth doing, this is the cell")
    print("that teaches prompt format as an engineering variable.\n")
    for name, p in PROMPTS.items():
        print(f"  --- {name} " + "-" * (60 - len(name)))
        for line in p.splitlines():
            print(f"    {line}")
    print(f"\n  ground truth: {17*23}")

  prompt style     parsed         failure   raw (truncated)
  ------------------------------------------------------------------------------
  bare               None       ambiguous   '17 * 23 equals 391.'
  format              391                   '391'
  tagged              391                   'ANSWER: 391'
  fenced+cot            1                   'To solve 17 * 23, we can use t'

  ground truth: 391


### ⚠️ Two traps this cell demonstrates

**Trap 1: the format instruction is doing the work, not the model's reasoning.**
If `bare` fails to parse and `tagged` succeeds with the same underlying answer, you
have not made the model smarter. You have made it *measurable*. Do not confuse
those in your write-up.

**Trap 2: you must fix the prompt before you collect data.**
Trying four prompts and reporting the best one is p-hacking. Legitimate procedure:
pick your prompt on a small **development set**, freeze it, then run the real
benchmark. Say in your report which prompt you used and how you chose it.

> **Give both sides the same care.** If you tuned your prompt for an hour and used
> a default heuristic for the classical side, your comparison is unfair, and the
> honesty section of the Duel Scorecard exists for exactly this admission.

## 5 · Reproducibility — scorecard axis 5

Same input, same output? Test it rather than assuming. Note `use_cache=False`:
the cache would hide the very thing we are measuring.

In [9]:
PROBE = ("List three uses of a paperclip. "
         "Reply with exactly three comma-separated items and nothing else.")

def variability(llm, prompt, n=3, temperature=0.0, use_cache=False):
    outs = []
    for i in range(n):
        r = llm.complete(prompt, temperature=temperature, seed=i,
                         use_cache=use_cache)
        outs.append(r.text.strip())
    return outs

if llm.backend in ("ollama", "api"):    # same as AUTOMATED, defined below
    print(f"temperature = 0.0, varying seed:")
    outs = variability(llm, PROBE, n=3, temperature=0.0)
    for i, o in enumerate(outs):
        print(f"  run {i}: {o[:70]!r}")
    print(f"  distinct answers: {len(set(outs))} of {len(outs)}")

    print(f"\ntemperature = 1.0, varying seed:")
    outs_hot = variability(llm, PROBE, n=3, temperature=1.0)
    for i, o in enumerate(outs_hot):
        print(f"  run {i}: {o[:70]!r}")
    print(f"  distinct answers: {len(set(outs_hot))} of {len(outs_hot)}")
else:
    print("Skipped: needs an automated backend (ollama or api).")
    print("""
On the manual backend, do this ONCE by hand: send the same prompt three times in
a fresh chat each time and record whether the answers differ. Report n=3 and note
the method. That is a legitimate measurement.""")

print("""
WHAT TO REPORT for axis 5: "5 runs, k distinct answers, at temperature T."
A system that returns 4 different answers to the same question has a property your
A* implementation does not, and that belongs in the table.""")

temperature = 0.0, varying seed:
  run 0: 'Organize papers, Align papers in folders, Secure loose papers'
  run 1: 'Organize papers, Align papers in folders, Secure loose papers'
  run 2: 'Organize papers, Align papers in folders, Secure loose papers'
  distinct answers: 1 of 3

temperature = 1.0, varying seed:
  run 0: 'Organize papers keep cables tidy store small items'
  run 1: 'Hold papers together fasten files together retain unfolded documents i'
  run 2: 'Organize papers, Align papers, Secure papers'
  distinct answers: 3 of 3

WHAT TO REPORT for axis 5: "5 runs, k distinct answers, at temperature T."
A system that returns 4 different answers to the same question has a property your
A* implementation does not, and that belongs in the table.


## 6 · Your first real comparison

The task: **sort a list of integers.** Chosen deliberately. Trivial classically,
verifiable exactly, and it scales, so you can see a cliff.

Three ingredients every duel needs:

| Ingredient | Why |
|---|---|
| A **classical baseline** | the control group |
| A **sound verifier** | so correctness is checked, not eyeballed |
| **Several problem sizes** | one point is not a curve (axis 6) |

In [10]:
import random
from dataclasses import dataclass

@dataclass
class SortInstance:
    id: str
    size: int
    items: list

def make_instances(sizes=(5, 10, 20, 40), per_size=5, seed=20250806):
    rng = random.Random(seed)
    out = []
    for n in sizes:
        for k in range(per_size):
            items = [rng.randint(0, 999) for _ in range(n)]
            out.append(SortInstance(id=f"n{n}-{k}", size=n, items=items))
    return out

INSTANCES = make_instances()
print(f"{len(INSTANCES)} instances across sizes "
      f"{sorted({i.size for i in INSTANCES})}")
print(f"example: {INSTANCES[0].id} -> {INSTANCES[0].items}")

20 instances across sizes [5, 10, 20, 40]
example: n5-0 -> [330, 235, 709, 207, 276]


In [11]:
# --- the classical system -------------------------------------------------
def classical_sort(inst):
    return sorted(inst.items)


# --- the LLM system -------------------------------------------------------
def make_llm_sort(llm):
    def llm_sort(inst):
        prompt = (
            "Sort this list of integers in ascending order.\n"
            f"List: {inst.items}\n"
            "Reply with ONLY the sorted list as comma-separated integers "
            "inside a fenced code block. No explanation."
        )
        text = llm.complete(prompt).text
        return parse_int_list(text)
    return llm_sort


def parse_int_list(text):
    """Prose -> list[int], or None. Same discipline as parse_int."""
    if not text or not text.strip():
        return None
    fence = re.search(r"```(?:\w+)?\s*(.*?)```", text, re.DOTALL)
    body = fence.group(1) if fence else text
    nums = re.findall(r"-?\d+", body)
    return [int(x) for x in nums] if nums else None


# --- the VERIFIER: this is the important part -----------------------------
def verify_sort(inst, answer):
    """Sound check. Returns (correct, failure_mode).

    Note it checks three separate things. An LLM can fail any of them
    independently, and lumping them together as 'wrong' throws away the most
    interesting part of your data.
    """
    if answer is None:
        return False, "malformed"
    if len(answer) != len(inst.items):
        return False, "invalid"                      # dropped or invented items
    if sorted(answer) != sorted(inst.items):
        return False, "invalid"                      # changed the multiset
    if any(answer[i] > answer[i+1] for i in range(len(answer)-1)):
        return False, "wrong-but-confident"          # right items, wrong order
    return True, None


# Sanity-check the verifier itself, on cases where we know the answer.
probe = SortInstance("t", 3, [3, 1, 2])
for ans, expect in [([1,2,3], True), ([3,2,1], False), ([1,2], False),
                    ([1,2,4], False), (None, False)]:
    ok, mode = verify_sort(probe, ans)
    assert ok == expect, (ans, ok, expect)
    print(f"  verify({str(ans):<10}) -> {str(ok):<6} {mode or ''}")
print("\nVerifier agrees with all five known cases. NOW it can be trusted.")

  verify([1, 2, 3] ) -> True   
  verify([3, 2, 1] ) -> False  wrong-but-confident
  verify([1, 2]    ) -> False  invalid
  verify([1, 2, 4] ) -> False  invalid
  verify(None      ) -> False  malformed

Verifier agrees with all five known cases. NOW it can be trusted.


### ⚠️ Test your verifier before you trust it

The cell above asserts the verifier against five cases with known answers. Do this
every time.

A broken verifier does not crash. **It silently produces a plausible number that
you then publish**. It is the same lesson as week 14's gradient check and week 9's
plan validator, and this is the first of the three times you will meet it.

In [12]:
from aicourse.compare import run_comparison, print_summary, scorecard_stub

systems = {"sorted()": classical_sort}
if AUTOMATED:
    systems["LLM"] = make_llm_sort(llm)
else:
    print("Manual backend: the classical arm runs below. For the LLM arm, do")
    print("FOUR instances by hand -- one per size -- and add those rows to your")
    print("table. Report n=4 and say so in the write-up.\n")
    print("Prompts to send (one per size):")
    for size in sorted({i.size for i in INSTANCES}):
        inst = next(i for i in INSTANCES if i.size == size)
        print(f"\n  --- {inst.id} " + "-" * 52)
        print(f"    Sort this list of integers in ascending order.")
        print(f"    List: {inst.items}")
        print(f"    Reply with ONLY the sorted list as comma-separated integers")
        print(f"    inside a fenced code block. No explanation.")
    print()

results = run_comparison(
    systems, INSTANCES,
    verifier=verify_sort,
    size_of=lambda i: i.size,
    id_of=lambda i: i.id,
    progress=False,
)
print_summary(results)


  system              n  solved    rate   median      p95
  -------------------------------------------------------
  sorted()           20      20   100%    0.00s    0.00s
  LLM                20       2    10%    0.00s  120.01s

  failure modes:
    LLM             invalid=10, malformed=3, wrong-but-confident=5

  solve rate by size  (axis 6 — look for the cliff)
                         5      10      20      40
  ------------------------------------------------
  sorted()           100%    100%    100%    100% 
  LLM                 40%      0%      0%      0% 


### Reading your own table

Whatever numbers you got, answer these four in your submission:

1. **Did the LLM's solve rate fall as the list got longer?** That is the cliff, and
   it is the characteristic finding of this course.
2. **Which failure mode dominated?** `invalid` (dropped or invented numbers) is a
   *different* problem from `wrong-but-confident` (right numbers, wrong order), and
   they suggest different fixes.
3. **What did each solved instance cost?** Compare seconds per instance. Then note
   that `sorted()` is O(n log n) with a proof, and the LLM has no such claim.
4. **Where might you have been unfair?**

> If your LLM scored 0% everywhere, that is a **result**, not a broken lab. Small
> local models often cannot do this reliably at n=40. Report it and move on.

In [13]:
print(scorecard_stub(results))

| Axis | sorted() | LLM |
|---|---|---|
| Correctness | 20/20 (100%) | 2/20 (10%) |
| Guarantee | **TODO — state it as a conditional** | **TODO — state it as a conditional** |
| Cost | **TODO** | **TODO** |
| Latency | 0.00s / 0.00s | 0.00s / 120.01s |
| Reproducibility | **TODO — run 5x, count distinct answers** | **TODO — run 5x, count distinct answers** |
| Scaling | 100% → 100% → 100% → 100% | 40% → 0% → 0% → 0% |
| Interpretability | **TODO — can you extract a certificate?** | **TODO — can you extract a certificate?** |
| Failure mode | none observed | invalid=10, malformed=3, wrong-but-confident=5 |

### Where we may have been unfair

**TODO — this section is worth real credit. Address at least three:**

- Did both systems get the same information?
- Did you tune one side's parameters but use a default prompt for the other?
- Is your instance distribution accidentally favourable to one side?
- Did you count the time you spent writing the prompt? The heuristic?
- Would a larger mo

### The four TODOs are not laziness

The harness fills in what it can *measure*. The four it leaves blank (guarantee,
reproducibility, interpretability, and the honesty section) are **judgements**.

A harness that guessed them would be teaching you that those axes are automatic,
and they are the ones that matter most. Axis 2 in particular:

> `sorted()` returns a permutation of the input in non-decreasing order, **for
> every input**, in O(n log n). *No condition, no exceptions.*
>
> The LLM produced a correct sort on 12/20 instances. **No claim is made about the
> 21st.**

Those are not the same kind of sentence, and telling them apart is the entire
subject of this course.

## 7 · The pattern you will use for fourteen weeks

```
        ┌─────────────┐
        │  instances  │   several sizes, fixed seed
        └──────┬──────┘
               │
      ┌────────┴────────┐
      ▼                 ▼
┌───────────┐    ┌─────────────┐
│ classical │    │ LLM + parse │   ← prose becomes data here
└─────┬─────┘    └──────┬──────┘
      │                 │
      └────────┬────────┘
               ▼
        ┌─────────────┐
        │  VERIFIER   │   ← sound, tested, and trusts nobody
        └──────┬──────┘
               ▼
     scorecard + scaling curve + honesty section
```

Every duel is this diagram with a different box in the middle. **You have now built
all of it once.**

## 8 · Your Week 0 deliverable

Create `week00/` in your course repo with:

**1 · `doctor.txt`**: the output of `python -m aicourse.doctor`.

**2 · `first_comparison.md`**: the summary table and the scorecard stub from this
notebook, with **all four TODOs filled in** and the four questions from §6
answered. Two or three sentences each is plenty.

**3 · One Failure Atlas entry** in the following format:
```markdown
### [Wk0] <one-line title>
**Setup:** what you asked, which backend and model
**Classical:** what the correct answer was, and how you knew
**LLM:** what it actually produced (paste it)
**Category:** wrong-but-confident | malformed | invalid | refused | timeout
**Why it matters:** one sentence, what would break if you shipped this
```

**4 · `AI_LOG.md`**: start it now, per
[`START-HERE.md`](START-HERE.md) (§ *Using AI in this course*). If you used an assistant
on this notebook, that is fine and it goes in the log. **Week 0 is the easiest week
of the semester to build that habit.**

This is worth a small amount of credit, and it is how the
instructor knows whose toolchain is broken *before* the first studio.

## 9 · If something is broken

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: aicourse` | Run Jupyter from the `cmp4004-week0/` folder (the one containing `aicourse/`) |
| Ollama: `connection refused` | `ollama serve` in another terminal |
| Ollama: `model not found` | `ollama pull qwen2.5:3b` (or `:1.5b` on ≤8 GB RAM) |
| Model is unbearably slow | Use `qwen2.5:1.5b`; cut `per_size` to 2; the cache means you pay once |
| Nothing works at all | `LLM(backend="manual")`. **This is a supported path, not a failure.** |

**Ask early.** A toolchain problem posted on Thursday is fifteen minutes of
someone's help; the same problem at 11 pm before the deadline is a lost grade.


In [14]:
fallo = next(r for r in results if r.system == "LLM" and r.failure_mode == "invalid")
inst = next(i for i in INSTANCES if i.id == fallo.instance_id)
print("Lista original:", inst.items)
print("Respuesta LLM:", fallo.answer)

Lista original: [479, 55, 711, 200, 750, 426, 950, 824, 707, 7]
Respuesta LLM: [7, 200, 426, 479, 55, 707, 711, 750, 824, 950, 479, 55, 711, 750, 426, 950, 824, 707, 7]
